# Reproducible Fruit Feature-Extraction and Classification Pipeline

This notebook is aligned with the cleaned preprocessing workflow and is designed for reproducibility, memory-aware execution, structured logging, and centralized outputs.

In [1]:
# ============================================================
# IMPORT REQUIRED LIBRARIES
# ============================================================

import os
import gc
import time
import random
import logging
from pathlib import Path

import cv2
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow.keras.applications import VGG16, VGG19, ResNet50
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_preprocess_input
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess_input

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    cohen_kappa_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc
)
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression


In [2]:
# ============================================================
# CONFIGURATION
# ============================================================

# ============================================================
# PROJECT ROOT DETECTION-TEMP
# ============================================================

PROJECT_ROOT = Path.cwd()

# If notebook launched from notebooks folder
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

print(f"Project Root: {PROJECT_ROOT}")

# ============================================================
# DATA DIRECTORIES
# ============================================================

DATA_DIR = PROJECT_ROOT / "data" / "processed"

OUTPUT_DIR = PROJECT_ROOT / "outputs"

RESULTS_DIR = OUTPUT_DIR / "results"

REPORTS_DIR = OUTPUT_DIR / "reports"

PLOTS_DIR = OUTPUT_DIR / "plots"

MODELS_DIR = OUTPUT_DIR / "models"

LOGS_DIR = OUTPUT_DIR / "logs"

# ============================================================
# CREATE OUTPUT DIRECTORIES
# ============================================================

for directory in [
    OUTPUT_DIR,
    RESULTS_DIR,
    REPORTS_DIR,
    PLOTS_DIR,
    MODELS_DIR,
    LOGS_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)



#DATA_DIR = Path("data/processed")
#OUTPUT_DIR = Path("outputs")
#RESULTS_DIR = OUTPUT_DIR / "results"
#REPORTS_DIR = OUTPUT_DIR / "reports"
#PLOTS_DIR = OUTPUT_DIR / "plots"
#MODELS_DIR = OUTPUT_DIR / "models"
#LOGS_DIR = OUTPUT_DIR / "logs"

#for directory in [OUTPUT_DIR, RESULTS_DIR, REPORTS_DIR, PLOTS_DIR, MODELS_DIR, LOGS_DIR]:
    #directory.mkdir(parents=True, exist_ok=True)

DATASETS = ["Banana", "Guava"]

IMAGE_SIZE = (210, 210)
VALID_EXTENSIONS = (".jpg", ".jpeg", ".png")

FEATURE_EXTRACTORS = ["VGG16", "VGG19", "ResNet50"]
CLASSIFIERS = ["Linear_SVM", "Quadratic_SVM", "Cubic_SVM", "RandomForest", "LogisticRegression"]

RANDOM_SEED = 42
TEST_SIZE = 0.20
BATCH_SIZE = 8

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)


Project Root: C:\Users\abiba\Documents\SoftImpact


In [3]:
# ============================================================
# LOGGING CONFIGURATION
# ============================================================

LOG_FILE = LOGS_DIR / "feature_extraction_pipeline.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE, mode="w", encoding="utf-8"),
        logging.StreamHandler()
    ],
    force=True
)

logging.info("Logging initialized.")
logging.info(f"TensorFlow version: {tf.__version__}")


2026-06-11 16:04:45,693 - INFO - Logging initialized.
2026-06-11 16:04:45,696 - INFO - TensorFlow version: 2.16.1


In [4]:
# ============================================================
# DATA LOADING
# ============================================================

def load_processed_images(dataset_directory):
    """
    Load processed images from a class-wise folder structure.

    Expected structure:
    data/processed/<Dataset>/<Class_Name>/<images>
    """

    images = []
    labels = []

    dataset_directory = Path(dataset_directory)

    if not dataset_directory.exists():
        raise FileNotFoundError(f"Dataset directory not found: {dataset_directory}")

    for class_folder in sorted(dataset_directory.iterdir()):

        if not class_folder.is_dir():
            continue

        class_name = class_folder.name
        logging.info(f"Loading class folder: {class_name}")

        for image_file in sorted(class_folder.iterdir()):

            if image_file.suffix.lower() not in VALID_EXTENSIONS:
                continue

            image = cv2.imread(str(image_file))

            if image is None:
                logging.warning(f"Skipping unreadable image: {image_file.name}")
                continue

            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

            if image.shape[:2] != IMAGE_SIZE:
                logging.warning(
                    f"Image {image_file.name} has shape {image.shape[:2]}, resizing to {IMAGE_SIZE}"
                )
                image = cv2.resize(image, IMAGE_SIZE)

            images.append(image)
            labels.append(class_name)

    if not images:
        raise ValueError(f"No images found in: {dataset_directory}")

    images = np.asarray(images, dtype=np.uint8)
    labels = np.asarray(labels)

    logging.info(f"Loaded {len(images)} images from {dataset_directory.name}")

    return images, labels


In [5]:
# ============================================================
# FEATURE EXTRACTOR FACTORIES
# ============================================================

def build_feature_extractor(name):
    """
    Return a pretrained CNN feature extractor and the matching preprocessing function.
    """

    if name == "VGG16":
        model = VGG16(
            weights="imagenet",
            include_top=False,
            pooling="avg",
            input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3)
        )
        return model, vgg_preprocess_input

    if name == "VGG19":
        model = VGG19(
            weights="imagenet",
            include_top=False,
            pooling="avg",
            input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3)
        )
        return model, vgg_preprocess_input

    if name == "ResNet50":
        model = ResNet50(
            weights="imagenet",
            include_top=False,
            pooling="avg",
            input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3)
        )
        return model, resnet_preprocess_input

    raise ValueError(f"Unknown feature extractor: {name}")


In [6]:
# ============================================================
# CLASSIFIER FACTORIES
# ============================================================

def build_classifier(name):
    """
    Return a classifier pipeline/estimator.
    SVMs and logistic regression are scaled for better numerical stability.
    """

    if name == "Linear_SVM":
        return Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", SVC(
                kernel="linear",
                probability=True,
                random_state=RANDOM_SEED
            ))
        ])

    if name == "Quadratic_SVM":
        return Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", SVC(
                kernel="poly",
                degree=2,
                probability=True,
                random_state=RANDOM_SEED
            ))
        ])

    if name == "Cubic_SVM":
        return Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", SVC(
                kernel="poly",
                degree=3,
                probability=True,
                random_state=RANDOM_SEED
            ))
        ])

    if name == "RandomForest":
        return RandomForestClassifier(
            n_estimators=200,
            random_state=RANDOM_SEED,
            n_jobs=-1
        )

    if name == "LogisticRegression":
        return Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(
                max_iter=5000,
                random_state=RANDOM_SEED
            ))
        ])

    raise ValueError(f"Unknown classifier: {name}")


In [7]:
# ============================================================
# BATCH-WISE FEATURE EXTRACTION
# ============================================================

def extract_features_batchwise(images, model, preprocess_fn, batch_size=BATCH_SIZE):
    """
    Extract flattened CNN features in batches to reduce peak memory usage.
    """

    feature_batches = []
    total_batches = int(np.ceil(len(images) / batch_size))

    logging.info(f"Starting feature extraction in {total_batches} batches.")

    for batch_index, start in enumerate(range(0, len(images), batch_size), start=1):
        end = min(start + batch_size, len(images))

        batch = images[start:end].astype(np.float32, copy=False)
        batch = preprocess_fn(batch)

        batch_features = model.predict(batch, verbose=0)
        batch_features = batch_features.reshape(batch_features.shape[0], -1)

        feature_batches.append(batch_features)

        if batch_index == 1 or batch_index == total_batches or batch_index % 10 == 0:
            logging.info(f"Processed feature batch {batch_index}/{total_batches}")

    features = np.concatenate(feature_batches, axis=0)

    logging.info(f"Feature extraction completed. Feature shape: {features.shape}")

    return features


In [8]:
# ============================================================
# EVALUATION AND EXPORT UTILITIES
# ============================================================

def evaluate_predictions(y_true, y_pred):
    """
    Compute standard classification metrics.
    """

    metrics = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision_Weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "Recall_Weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "F1_Weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "Kappa": cohen_kappa_score(y_true, y_pred)
    }

    report_dict = classification_report(
        y_true,
        y_pred,
        output_dict=True,
        zero_division=0
    )

    report_text = classification_report(
        y_true,
        y_pred,
        zero_division=0
    )

    cm = confusion_matrix(y_true, y_pred)

    return metrics, report_dict, report_text, cm


def save_classification_report(report_dict, save_path):
    """
    Save sklearn classification_report output_dict as CSV.
    """

    report_df = pd.DataFrame(report_dict).transpose()
    report_df.to_csv(save_path)


def save_confusion_matrix(cm, class_names, title, save_path):
    """
    Save confusion matrix plot.
    """

    plt.figure(figsize=(7, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names
    )
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()


def save_multiclass_roc(y_true, y_proba, class_names, title, save_path):
    """
    Save one-vs-rest ROC curves and return macro-average AUC.
    """

    y_true_bin = label_binarize(y_true, classes=np.arange(len(class_names)))

    plt.figure(figsize=(8, 6))
    auc_values = []

    for class_index, class_name in enumerate(class_names):
        if len(np.unique(y_true_bin[:, class_index])) < 2:
            logging.warning(f"Skipping ROC for class {class_name} due to insufficient class variation.")
            continue

        fpr, tpr, _ = roc_curve(y_true_bin[:, class_index], y_proba[:, class_index])
        class_auc = auc(fpr, tpr)
        auc_values.append(class_auc)

        plt.plot(
            fpr,
            tpr,
            lw=2,
            label=f"{class_name} (AUC = {class_auc:.3f})"
        )

    plt.plot([0, 1], [0, 1], "k--", lw=1)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()

    if len(auc_values) == 0:
        return np.nan

    return float(np.mean(auc_values))


In [9]:
# ============================================================
# COMPLETE DATASET PIPELINE
# ============================================================

def process_dataset(dataset_name):
    """
    Run feature extraction + classification pipeline for a single dataset.
    """

    logging.info("=" * 80)
    logging.info(f"Starting dataset: {dataset_name}")
    logging.info("=" * 80)

    dataset_path = DATA_DIR / dataset_name
    images, labels = load_processed_images(dataset_path)

    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(labels)

    dataset_model_dir = MODELS_DIR / dataset_name
    dataset_model_dir.mkdir(parents=True, exist_ok=True)

    # Save label encoder for reproducibility
    joblib.dump(label_encoder, dataset_model_dir / "label_encoder.joblib")

    results = []

    for extractor_name in FEATURE_EXTRACTORS:

        logging.info("-" * 80)
        logging.info(f"Feature extractor: {extractor_name}")
        logging.info("-" * 80)

        extractor_model, preprocess_fn = build_feature_extractor(extractor_name)

        extract_start = time.perf_counter()
        features = extract_features_batchwise(
            images=images,
            model=extractor_model,
            preprocess_fn=preprocess_fn,
            batch_size=BATCH_SIZE
        )
        extract_time = time.perf_counter() - extract_start

        X_train, X_test, y_train, y_test = train_test_split(
            features,
            y,
            test_size=TEST_SIZE,
            random_state=RANDOM_SEED,
            stratify=y
        )

        logging.info(
            f"Train/Test split completed for {extractor_name}: "
            f"X_train={X_train.shape}, X_test={X_test.shape}"
        )

        extractor_model_dir = dataset_model_dir / extractor_name
        extractor_model_dir.mkdir(parents=True, exist_ok=True)

        for classifier_name in CLASSIFIERS:

            logging.info(f"Training classifier: {classifier_name}")

            classifier = build_classifier(classifier_name)

            train_start = time.perf_counter()
            classifier.fit(X_train, y_train)
            train_time = time.perf_counter() - train_start

            predict_start = time.perf_counter()
            y_pred = classifier.predict(X_test)
            y_proba = classifier.predict_proba(X_test)
            predict_time = time.perf_counter() - predict_start

            metrics, report_dict, report_text, cm = evaluate_predictions(y_test, y_pred)

            roc_path = PLOTS_DIR / f"{dataset_name}_{extractor_name}_{classifier_name}_roc.png"
            roc_auc_macro = save_multiclass_roc(
                y_true=y_test,
                y_proba=y_proba,
                class_names=label_encoder.classes_,
                title=f"{dataset_name} - {extractor_name} - {classifier_name} ROC",
                save_path=roc_path
            )

            cm_path = PLOTS_DIR / f"{dataset_name}_{extractor_name}_{classifier_name}_confusion_matrix.png"
            save_confusion_matrix(
                cm=cm,
                class_names=label_encoder.classes_,
                title=f"{dataset_name} - {extractor_name} - {classifier_name} Confusion Matrix",
                save_path=cm_path
            )

            report_csv_path = REPORTS_DIR / f"{dataset_name}_{extractor_name}_{classifier_name}_classification_report.csv"
            save_classification_report(report_dict, report_csv_path)

            report_txt_path = REPORTS_DIR / f"{dataset_name}_{extractor_name}_{classifier_name}_classification_report.txt"
            with open(report_txt_path, "w", encoding="utf-8") as f:
                f.write(report_text)

            model_path = extractor_model_dir / f"{classifier_name}.joblib"
            joblib.dump(classifier, model_path)

            row = {
                "Dataset": dataset_name,
                "Feature_Extractor": extractor_name,
                "Classifier": classifier_name,
                "Extract_Time_Seconds": extract_time,
                "Train_Time_Seconds": train_time,
                "Predict_Time_Seconds": predict_time,
                "Accuracy": metrics["Accuracy"],
                "Precision_Weighted": metrics["Precision_Weighted"],
                "Recall_Weighted": metrics["Recall_Weighted"],
                "F1_Weighted": metrics["F1_Weighted"],
                "Kappa": metrics["Kappa"],
                "ROC_AUC_Macro": roc_auc_macro,
                "Model_Path": str(model_path),
                "Confusion_Matrix_Path": str(cm_path),
                "ROC_Path": str(roc_path),
                "Report_CSV_Path": str(report_csv_path),
                "Report_TXT_Path": str(report_txt_path)
            }
            results.append(row)

            logging.info(
                f"Completed {dataset_name} | {extractor_name} | {classifier_name} "
                f"-> Accuracy: {metrics['Accuracy']:.4f}, Kappa: {metrics['Kappa']:.4f}"
            )

            del classifier, y_pred, y_proba, metrics, report_dict, report_text, cm
            gc.collect()

        del extractor_model, features, X_train, X_test, y_train, y_test
        tf.keras.backend.clear_session()
        gc.collect()

    results_df = pd.DataFrame(results)
    results_csv = RESULTS_DIR / f"{dataset_name}_results.csv"
    results_df.to_csv(results_csv, index=False)

    logging.info(f"Saved dataset results: {results_csv}")
    logging.info(f"Completed dataset: {dataset_name}")

    return results_df


In [10]:
# ============================================================
# MAIN EXECUTION
# ============================================================

def main():
    """
    Run the full reproducible pipeline for all datasets.
    """

    all_results = []

    for dataset_name in DATASETS:
        dataset_results = process_dataset(dataset_name)
        all_results.append(dataset_results)

    final_results = pd.concat(all_results, ignore_index=True)
    final_results_csv = RESULTS_DIR / "final_results.csv"
    final_results.to_csv(final_results_csv, index=False)

    logging.info("=" * 80)
    logging.info(f"All datasets completed. Final results saved to: {final_results_csv}")
    logging.info("=" * 80)


if __name__ == "__main__":
    main()


2026-06-11 16:04:45,951 - INFO - ================================================================================
2026-06-11 16:04:45,952 - INFO - Starting dataset: Banana
2026-06-11 16:04:45,953 - INFO - ================================================================================
2026-06-11 16:04:45,954 - INFO - Loading class folder: Class_A
2026-06-11 16:04:46,802 - INFO - Loading class folder: Class_B
2026-06-11 16:04:47,254 - INFO - Loading class folder: Defect
2026-06-11 16:04:48,058 - INFO - Loaded 1203 images from Banana
2026-06-11 16:04:48,066 - INFO - --------------------------------------------------------------------------------
2026-06-11 16:04:48,068 - INFO - Feature extractor: VGG16
2026-06-11 16:04:48,069 - INFO - --------------------------------------------------------------------------------
2026-06-11 16:04:48,805 - INFO - Starting feature extraction in 151 batches.
2026-06-11 16:04:51,785 - INFO - Processed feature batch 1/151
2026-06-11 16:05:19,469 - INFO - Pro

2026-06-11 16:14:03,661 - WARNING - From C:\Users\abiba\anaconda3\Lib\site-packages\keras\src\backend\common\global_state.py:82: The name tf.reset_default_graph is deprecated. Please use tf.compat.v1.reset_default_graph instead.

2026-06-11 16:14:04,193 - INFO - --------------------------------------------------------------------------------
2026-06-11 16:14:04,200 - INFO - Feature extractor: VGG19
2026-06-11 16:14:04,204 - INFO - --------------------------------------------------------------------------------
2026-06-11 16:14:05,326 - INFO - Starting feature extraction in 151 batches.
2026-06-11 16:14:10,387 - INFO - Processed feature batch 1/151
2026-06-11 16:14:52,472 - INFO - Processed feature batch 10/151
2026-06-11 16:15:37,604 - INFO - Processed feature batch 20/151
2026-06-11 16:16:22,698 - INFO - Processed feature batch 30/151
2026-06-11 16:17:07,848 - INFO - Processed feature batch 40/151
2026-06-11 16:17:53,716 - INFO - Processed feature batch 50/151
2026-06-11 16:18:39,829 